# IOG ㈜K-Bottle 분석 노트북

수요예측 → 생산 스케줄링 → 생산계획 → 전략 시뮬레이션 순서로 의사결정 근거를 정리한다.
모든 로직은 `iog/*.py` 모듈에 있고, 이 노트북은 호출·시각화만 한다.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'; plt.rcParams['axes.unicode_minus'] = False
from data_loader import load_demand
from forecast import backtest, choose_model
import scheduler
from production_plan import critical_ratio, est_makespan
from cost_model import production_cost
from simulate import compare
data = load_demand()

## 1. 수요 데이터
P1 = 삼성전자 종가(거래일만), P2 = BTC 종가(매일). 2018-05-04 액면분할 이전 P1은 제외.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=False)
for a, (p, s) in zip(ax, data.items()):
    s[s.index >= '2025-06-01'].plot(ax=a, title=f'{p} 일별 수요'); a.set_ylabel('개')
plt.tight_layout()

## 2. 수요예측 모델 백테스트
Rolling origin 250회, horizon 1~10. 주가·코인은 random walk에 가까워 naive/drift 계열이 복잡한 모델과 비슷하거나 낫다. (ARIMA는 계산량 때문에 최근 80회만 평가)

In [ ]:
bt = {}
for p, s in data.items():
    bt[p], _ = backtest(s)
    best, avg = choose_model(bt[p])
    print(p, '선택:', best); display(avg.round(2).to_frame('평균 MAPE(%) h=1..8'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, p in zip(ax, bt):
    bt[p].pivot(index='h', columns='model', values='mape').plot(ax=a, title=f'{p} horizon별 MAPE(%)')
plt.tight_layout()

## 3. 생산 스케줄링 (20설비 Permutation Flow Shop)
매뉴얼 예제(월요일 Job 1~5) 검증 후, 실제 Job 수(P1≈275, P2≈86)에서 SPT(게임 Auto) vs NEH vs IG 비교.

In [ ]:
P = np.array([[67,83,32,90,29,1,3,69,86,7,30,93,93,50,26,37,60,61,24,2],[36,3,16,43,62,92,55,66,86,83,34,79,20,14,47,67,55,79,70,85],
              [11,92,22,68,37,69,73,34,86,54,13,58,54,85,50,21,73,55,25,31],[30,66,61,4,58,48,13,53,78,63,53,19,23,59,56,20,58,90,51,43],
              [18,10,38,47,18,4,10,63,38,80,85,24,4,98,78,97,32,87,73,56]], np.int64)
for name, s in [('ID 1-2-3-4-5', [1,2,3,4,5]), ('SPT 3-5-4-2-1', [3,5,4,2,1]), ('최적 5-2-4-3-1', [5,2,4,3,1])]:
    print(f'{name:16s} makespan = {scheduler.makespan(P, np.array(s) - 1)}')

In [ ]:
rows = []
for prod, n in (('P2', 86), ('P1', 275)):
    p = scheduler.load_times(0)[:n]
    spt = scheduler.makespan(p, scheduler.spt(p)); neh = scheduler.makespan(p, scheduler.neh(p))
    _, ig = scheduler.iterated_greedy(p, time_limit=15)
    for algo, ms in [('SPT(게임 Auto)', spt), ('NEH', neh), ('IG 15초', ig)]:
        rows.append(dict(product=prod, jobs=n, algo=algo, makespan=ms, 생산비=production_cost(prod, ms, n)))
sched = pd.DataFrame(rows); sched

In [ ]:
def gantt(p, seq, title):
    m = p.shape[1]; c = np.zeros((len(seq), m))
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, j in enumerate(seq):
        for k in range(m):
            start = max(c[i-1, k] if i else 0, c[i, k-1] if k else 0)
            c[i, k] = start + p[j, k]
            ax.barh(k, p[j, k], left=start, color=plt.cm.tab20(i % 20), edgecolor='none')
    ax.invert_yaxis(); ax.set_yticks(range(m), [f'M{k+1}' for k in range(m)])
    ax.set_title(f'{title} — makespan {int(c[-1, -1])}'); ax.set_xlabel('시간')
p = scheduler.load_times(0)[:30]
gantt(p, scheduler.spt(p), 'SPT (30 jobs)'); gantt(p, scheduler.iterated_greedy(p, 5)[0], 'IG (30 jobs)')

## 4. 생산비 구조와 배치 결정
Makespan이 Job 수에 거의 선형이라 배치를 합쳐도 인건비 절감이 없다. 오히려 정규시간(7,000)이 배치마다 주어지므로 **매일 생산**이 유리하다 (DP로 확인).

In [ ]:
ns = np.arange(10, 501, 10)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ms = [est_makespan(0, n) for n in ns]
ax[0].plot(ns, ms); ax[0].axhline(7000, ls='--', c='r', label='정규시간 기준 7,000'); ax[0].legend(); ax[0].set(title='Job 수 vs NEH makespan (월)', xlabel='jobs')
ax[1].plot(ns, [production_cost('P1', m, n) / (n * 1000) for m, n in zip(ms, ns)])
ax[1].set(title='P1 개당 생산비(셋업+인건비)', xlabel='jobs', ylabel='원/개'); plt.tight_layout()
print({p: round(critical_ratio(p), 3) for p in ('P1', 'P2')}, '← 이론 newsvendor 비율 (1일 보유 가정)')

## 5. 전략 시뮬레이션 (2026-08-02 ~ 10-03, 9주)
매주(또는 매일) 그 시점까지의 데이터만으로 계획 → 실제 수요로 정산. 자재는 충분하다고 가정.
- 게임 Auto: MA5 예측 + SPT
- 최적예측+NEH: 백테스트 최적 모델 + NEH 스케줄, 분위수 q로 생산
- replan=daily: 평일마다 전날까지 데이터로 재계획 (폼 덮어쓰기)

In [ ]:
sim = compare(data, '2026-08-02', 9)
cols = ['revenue', 'production', 'material', 'holding', 'stockout_cost', 'profit']
view = sim.copy(); view[cols] = (view[cols] / 1e6).round(0)
view

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, p in zip(ax, ('P1', 'P2')):
    s = sim[(sim['product'] == p) & (sim.strategy == '최적예측+NEH')]
    for r, g in s.groupby('replan'):
        a.plot(g.q, g.profit / 1e6, marker='o', label=f'최적예측+NEH ({r})')
    base = sim[(sim['product'] == p) & (sim.strategy.str.startswith('게임'))].profit.iloc[0] / 1e6
    a.axhline(base, c='gray', ls='--', label='게임 Auto'); a.set(title=f'{p} 9주 이익(백만원) vs 생산 분위수', xlabel='q'); a.legend()
plt.tight_layout()

## 6. 결론
1. **스케줄링**: IG가 SPT 대비 makespan을 크게 줄여 잔업비 절감 — 가장 확실한 이익원.
2. **예측**: random walk 성격 → naive/drift가 최선. 폼에는 중앙값 입력.
3. **생산량**: 주 1회 계획이면 q≈0.55, 매일 재계획이면 q≈0.75가 최적 (`config.SERVICE_Q`).
4. **배치**: 매일 생산이 유리 (보유비 30원/개·일 > 셋업 절감, 정규시간은 배치마다).
5. **할인**: 평소 0, 라운드 마지막 날 재고 소진용으로만.